# DistilBERT + LoRA for Financial Sentiment Classification

Каркас эксперимента для честного сравнения с `DistilBERT` full fine-tuning. Ноутбук использует фиксированные разбиения и общие константы проекта; обучение в этой версии не запускается.

## Импорты и настройка окружения

In [ ]:
import inspect
import json
import time

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

from src.constants import (
    BASE_MODEL_DISTILBERT,
    DATASET_NAME,
    FINAL_TEST_EVALUATION_ONLY,
    MAX_LENGTH,
    PROJECT_ROOT,
    REPORTS_TABLES_DIR,
    SEED,
    SELECTION_METRIC,
    TEST_CSV_PATH,
    TRAIN_CSV_PATH,
    VALIDATION_CSV_PATH,
    id2label,
    label2id,
)

RUN_NAME = "distilbert_lora"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / RUN_NAME
CHECKPOINTS_DIR = ARTIFACTS_DIR / "checkpoints"
ADAPTER_DIR = ARTIFACTS_DIR / "adapter"
RUN_TABLES_DIR = REPORTS_TABLES_DIR / RUN_NAME

## Загрузка фиксированных разбиений данных

In [ ]:
train_df = pd.read_csv(TRAIN_CSV_PATH).reset_index(drop=True)
validation_df = pd.read_csv(VALIDATION_CSV_PATH).reset_index(drop=True)
test_df = pd.read_csv(TEST_CSV_PATH).reset_index(drop=True)

for split_df in (train_df, validation_df, test_df):
    split_df["label"] = split_df["label"].astype(int)

raw_dataframes = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

raw_datasets = DatasetDict({
    split_name: Dataset.from_pandas(split_df, preserve_index=False)
    for split_name, split_df in raw_dataframes.items()
})
raw_datasets = raw_datasets.rename_column("label", "labels")

raw_datasets

## Токенизация текста

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DISTILBERT)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_datasets = raw_datasets.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text", "label_name"],
)
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

sample = tokenized_datasets["train"][0]
print("input_ids shape:", sample["input_ids"].shape)
print("attention_mask shape:", sample["attention_mask"].shape)
print("label:", sample["labels"].item())

## Метрики качества

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
    }


def softmax_numpy(logits):
    shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
    probabilities = np.exp(shifted_logits)
    return probabilities / np.sum(probabilities, axis=1, keepdims=True)


def main_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted")),
    }


assert SELECTION_METRIC == "macro_f1"
assert SELECTION_METRIC in compute_metrics((np.zeros((1, len(id2label))), np.array([0])))

## Базовая модель DistilBERT

In [ ]:
NUM_LABELS = len(id2label)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_DISTILBERT,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

modules_to_save = [
    module_name
    for module_name in ("pre_classifier", "classifier")
    if hasattr(base_model, module_name)
]
assert modules_to_save == ["pre_classifier", "classifier"], (
    "Expected DistilBERT classification head modules were not found: "
    f"{modules_to_save}"
)

print("Base model:", BASE_MODEL_DISTILBERT)
print("Classification head modules to train and save:", modules_to_save)

## Конфигурация LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_lin", "v_lin"],
    modules_to_save=modules_to_save,
)

model = get_peft_model(base_model, lora_config)
model.to(device)

def summarize_parameters(model, preview_size=20):
    total_params = sum(parameter.numel() for parameter in model.parameters())
    trainable_names = [
        name for name, parameter in model.named_parameters() if parameter.requires_grad
    ]
    trainable_params = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    trainable_share = 100 * trainable_params / total_params

    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Trainable share:      {trainable_share:.4f}%")
    print("Trainable parameter names (preview):")
    for name in trainable_names[:preview_size]:
        print(" -", name)

    return {
        "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_share_pct": trainable_share,
        "trainable_parameter_names": trainable_names,
    }

parameter_summary = summarize_parameters(model)
trainable_names = parameter_summary["trainable_parameter_names"]

assert any("lora_" in name for name in trainable_names), "No trainable LoRA parameters found."
for head_module in modules_to_save:
    assert any(head_module in name for name in trainable_names), (
        f"The classification head module {head_module!r} is not trainable."
    )

unexpected_trainable_names = [
    name
    for name in trainable_names
    if "lora_" not in name
    and not any(head_module in name for head_module in modules_to_save)
]
assert not unexpected_trainable_names, (
    "Unexpected trainable backbone parameters: "
    f"{unexpected_trainable_names[:10]}"
)

### Проверка forward pass без обучения

In [ ]:
model.eval()
smoke_batch_size = min(2, len(tokenized_datasets["validation"]))
smoke_batch = tokenized_datasets["validation"][:smoke_batch_size]
smoke_inputs = {
    key: smoke_batch[key].to(device)
    for key in ("input_ids", "attention_mask")
}

with torch.no_grad():
    smoke_output = model(**smoke_inputs)

expected_logits_shape = (smoke_batch_size, NUM_LABELS)
assert tuple(smoke_output.logits.shape) == expected_logits_shape, (
    f"Unexpected logits shape: {tuple(smoke_output.logits.shape)}; "
    f"expected {expected_logits_shape}."
)
print("Forward pass logits shape:", tuple(smoke_output.logits.shape))

## Обучение модели

In [ ]:
# The test split is intentionally absent from this training and selection block.
LEARNING_RATE = 2e-5
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
LOGGING_STEPS = 50

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)

training_args_kwargs = {
    "output_dir": str(CHECKPOINTS_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "logging_steps": LOGGING_STEPS,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "report_to": "none",
    "seed": SEED,
    "data_seed": SEED,
    "fp16": torch.cuda.is_available(),
}

training_arguments_signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in training_arguments_signature.parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)
assert training_args.metric_for_best_model == SELECTION_METRIC
assert training_args.load_best_model_at_end is True

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

training_start_time = time.perf_counter()
train_result = trainer.train()
training_time_sec = time.perf_counter() - training_start_time

assert trainer.state.best_model_checkpoint is not None
validation_metrics = trainer.evaluate(
    tokenized_datasets["validation"],
    metric_key_prefix="validation",
)

print(f"Training time: {training_time_sec:.2f} sec")
print("Best checkpoint selected on validation macro-F1:", trainer.state.best_model_checkpoint)
validation_metrics

## Оценка качества модели

In [ ]:
assert FINAL_TEST_EVALUATION_ONLY is True
assert trainer.state.best_model_checkpoint is not None, "Run training before final test evaluation."

# Trainer has already restored the checkpoint selected by validation macro-F1.
test_prediction_output = trainer.predict(
    tokenized_datasets["test"],
    metric_key_prefix="test",
)
test_logits = test_prediction_output.predictions
test_probabilities = softmax_numpy(test_logits)
test_predictions = np.argmax(test_probabilities, axis=1)
test_labels = raw_dataframes["test"]["label"].to_numpy()
test_metrics = main_metrics(test_labels, test_predictions)

class_names = [id2label[class_id] for class_id in sorted(id2label)]
test_classification_report = classification_report(
    test_labels,
    test_predictions,
    labels=sorted(id2label),
    target_names=class_names,
    output_dict=True,
    zero_division=0,
)
test_classification_report_df = (
    pd.DataFrame(test_classification_report).transpose().rename_axis("class").reset_index()
)

test_confusion_matrix = confusion_matrix(
    test_labels,
    test_predictions,
    labels=sorted(id2label),
)
test_confusion_matrix_df = pd.DataFrame(
    test_confusion_matrix,
    index=[f"true_{class_name}" for class_name in class_names],
    columns=[f"pred_{class_name}" for class_name in class_names],
)

test_predictions_df = raw_dataframes["test"].copy()
test_predictions_df["true_label"] = test_labels
test_predictions_df["true_label_name"] = test_predictions_df["true_label"].map(id2label)
test_predictions_df["pred_label"] = test_predictions
test_predictions_df["pred_label_name"] = test_predictions_df["pred_label"].map(id2label)
test_predictions_df["correct"] = test_predictions_df["true_label"] == test_predictions_df["pred_label"]
for class_id, class_name in id2label.items():
    test_predictions_df[f"prob_{class_name.lower()}"] = test_probabilities[:, class_id]

print("Final test metrics from the validation-selected checkpoint:")
test_metrics

## Сохранение артефактов

In [ ]:
RUN_TABLES_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

metrics_payload = {
    "run_name": RUN_NAME,
    "model": BASE_MODEL_DISTILBERT,
    "adaptation": "lora",
    "selection_metric": SELECTION_METRIC,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "training_time_sec": training_time_sec,
    "validation_accuracy": float(validation_metrics["validation_accuracy"]),
    "validation_macro_f1": float(validation_metrics["validation_macro_f1"]),
    "validation_weighted_f1": float(validation_metrics["validation_weighted_f1"]),
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "total_params": parameter_summary["total_params"],
    "trainable_params": parameter_summary["trainable_params"],
    "trainable_share": parameter_summary["trainable_share_pct"],
}

metrics_json_path = RUN_TABLES_DIR / f"{RUN_NAME}_metrics.json"
predictions_csv_path = RUN_TABLES_DIR / f"{RUN_NAME}_test_predictions.csv"
classification_report_path = RUN_TABLES_DIR / f"{RUN_NAME}_test_classification_report.csv"
confusion_matrix_path = RUN_TABLES_DIR / f"{RUN_NAME}_test_confusion_matrix.csv"
comparison_table_path = RUN_TABLES_DIR / f"{RUN_NAME}_comparison_row.csv"
training_arguments_path = RUN_TABLES_DIR / f"{RUN_NAME}_training_arguments.json"
experiment_config_path = RUN_TABLES_DIR / f"{RUN_NAME}_experiment_config.json"

with metrics_json_path.open("w", encoding="utf-8") as metrics_file:
    json.dump(metrics_payload, metrics_file, indent=2)

test_predictions_df.to_csv(predictions_csv_path, index=False)
test_classification_report_df.to_csv(classification_report_path, index=False)
test_confusion_matrix_df.to_csv(confusion_matrix_path)

comparison_row = {
    "model": "distilbert-base-uncased",
    "adaptation": "lora",
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "bearish_f1": float(test_classification_report["Bearish"]["f1-score"]),
    "bullish_f1": float(test_classification_report["Bullish"]["f1-score"]),
    "neutral_f1": float(test_classification_report["Neutral"]["f1-score"]),
    "total_params": parameter_summary["total_params"],
    "trainable_params": parameter_summary["trainable_params"],
    "trainable_share": parameter_summary["trainable_share_pct"],
    "training_time_sec": training_time_sec,
}
pd.DataFrame([comparison_row]).to_csv(comparison_table_path, index=False)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
training_args.to_json_file(str(training_arguments_path))

experiment_config = {
    "dataset_name": DATASET_NAME,
    "model": BASE_MODEL_DISTILBERT,
    "adaptation": "lora",
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "selection_metric": SELECTION_METRIC,
    "test_evaluation_after_validation_selection_only": FINAL_TEST_EVALUATION_ONLY,
    "data_paths": {
        "train": str(TRAIN_CSV_PATH),
        "validation": str(VALIDATION_CSV_PATH),
        "test": str(TEST_CSV_PATH),
    },
    "lora": {
        "task_type": "SEQ_CLS",
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.1,
        "bias": "none",
        "target_modules": ["q_lin", "v_lin"],
        "modules_to_save": modules_to_save,
    },
}
with experiment_config_path.open("w", encoding="utf-8") as config_file:
    json.dump(experiment_config, config_file, indent=2)

print("Saved table and JSON outputs to:", RUN_TABLES_DIR)
print("Saved LoRA adapter and tokenizer to:", ADAPTER_DIR)
pd.DataFrame([comparison_row])